# 04: End to end on a real donor file (KDD Cup 1998)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilanthroPy-Project/PhilanthroPy/blob/main/examples/notebooks/04_kdd98_end_to_end.ipynb)

Notebooks 01 to 03 run on synthetic data. This one runs on **95,412 real donors**:
the KDD Cup 1998 direct-mail file, with each donor's gift history across 22
earlier mailings and the outcome of one more mailing (`TARGET_B` = gave,
`TARGET_D` = how much).

It walks the whole path a development shop would take:

1. Size up the donor file (concentration, retention, lifetime value)
2. Turn a wide export into a gift log, clean it, and add fiscal years
3. Read the same gifts as a Raiser's Edge export, pledges and all
4. Build features **as of** the decision date
5. See what leakage does to a lapse model's backtest
6. Score who will respond, and explain the score
7. Predict gift size with an honest interval, and build an ask ladder
8. Decide who to mail, in dollars
9. Check the mailing list for group disparity
10. Save the model for next year

The first run downloads the dataset (about 35 MB) to `~/philanthropy_data`
once; later runs read the cached copy. Nothing about your own data is sent
anywhere.

> **Dataset terms.** Under the KDD Cup 1998 terms, teaching material must not
> name the organisation that supplied the data; it is cited here only as
> "KDD Cup 1998".

In [ ]:
# In Colab, install the library first (the Raiser's Edge reader needs 0.8+ or main):
# !pip install "git+https://github.com/PhilanthroPy-Project/PhilanthroPy.git"
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline

from philanthropy.datasets import fetch_kdd98_donors
from philanthropy.ingest import raisers_edge_gifts_to_features, read_raisers_edge_gifts
from philanthropy.inspection import donor_feature_importance
from philanthropy.metrics import (
    cost_per_dollar_raised,
    disparate_impact_ratio,
    donor_lifetime_value,
    donor_retention_rate,
    fundraising_roi,
    gift_concentration_gini,
    interval_report,
    selection_rate_by_group,
    top_donor_share,
)
from philanthropy.model_selection import FiscalYearGroupedSplitter
from philanthropy.models import (
    AskAmountRecommender,
    GiftIntervalCalibrator,
    LapsePredictor,
    MajorGiftClassifier,
)
from philanthropy.preprocessing import (
    CRMCleaner,
    FiscalYearTransformer,
    RFMTransformer,
    WealthScreeningImputer,
)
from philanthropy.utils import load_model, save_model
from philanthropy.visualisation import plot_affinity_distribution, plot_retention_waterfall

RANDOM_STATE = 0
donors = fetch_kdd98_donors()
donors.shape

## 1. Size up the donor file

Before any model: how concentrated is giving, and what does a donor look like?
`RAMNTALL` is each donor's lifetime giving up to the mailing being predicted.

In [ ]:
lifetime = donors["RAMNTALL"]
print(f"Donors:                         {len(donors):,}")
print(f"Lifetime giving, total:         ${lifetime.sum():,.0f}")
print(f"Gini of lifetime giving:        {gift_concentration_gini(lifetime):.3f}")
print(f"Share from the top 10% donors:  {top_donor_share(lifetime, top_fraction=0.10):.1%}")
print(f"Responded to the next mailing:  {donors['TARGET_B'].mean():.2%}")
print(f"Average gift when they did:     ${donors.loc[donors['TARGET_D'] > 0, 'TARGET_D'].mean():.2f}")

## 2. From a wide export to a clean gift log

The file is wide: one row per donor, with `RDATE_3`..`RDATE_24` (gift date,
encoded `YYMM`) and `RAMNT_3`..`RAMNT_24` (amount) for each earlier mailing.
Most CRM exports you meet are long instead: one row per gift. Reshape, then run
the two cleaning steps every analysis starts with.

In [ ]:
parts = []
for i in range(3, 25):
    promo = donors[["CONTROLN", f"RDATE_{i}", f"RAMNT_{i}"]].dropna()
    promo.columns = ["donor_id", "yymm", "gift_amount"]
    parts.append(promo)
gifts = pd.concat(parts, ignore_index=True)
gifts["gift_date"] = pd.to_datetime(
    "19" + gifts["yymm"].astype(int).astype(str).str.zfill(4), format="%Y%m"
)
gifts = gifts[["donor_id", "gift_date", "gift_amount"]]

gifts = (CRMCleaner(date_col="gift_date", amount_col="gift_amount")
         .set_output(transform="pandas").fit_transform(gifts)
         .astype({"donor_id": int}))
fiscal = (FiscalYearTransformer(date_col="gift_date", fiscal_year_start=7)
          .set_output(transform="pandas").fit_transform(gifts).astype(int))
gifts = pd.concat([gifts, fiscal], axis=1)
print(f"{len(gifts):,} gifts from {gifts['donor_id'].nunique():,} donors, "
      f"{gifts['gift_date'].min():%b %Y} to {gifts['gift_date'].max():%b %Y}")
gifts.head()

**Retention and lifetime value.** Compare the donors who gave in fiscal 1995
(July 1994 to June 1995) with those who gave in fiscal 1996. Read the number
with care: this file only contains people who gave between June 1995 and June
1996, so retention here runs higher than on a full donor file.

In [ ]:
by_fy = gifts.groupby("fiscal_year")["donor_id"].apply(set)
prior, current = by_fy[1995], by_fy[1996]
earlier = set().union(*[by_fy[fy] for fy in by_fy.index if fy < 1995])

retention = donor_retention_rate(current, prior)
lapsed = len(prior - current)
recovered = len((current - prior) & earlier)
acquired = len(current - prior - earlier)
print(f"Retention FY1995 to FY1996: {retention:.1%}")

avg_annual_gift = gifts[gifts["fiscal_year"] == 1996].groupby("donor_id")["gift_amount"].sum().mean()
ltv = donor_lifetime_value(avg_annual_gift, lifespan_years=5, retention_rate=retention)
print(f"Average annual giving per FY1996 donor: ${avg_annual_gift:.2f}")
print(f"5-year lifetime value at that retention: ${ltv:.2f}")

plot_retention_waterfall(len(prior), acquired, lapsed, recovered);

## 3. The same gifts as a Raiser's Edge export

Real shops hand over a CRM export, not a tidy table. Write the gift log out
under Raiser's Edge's own column labels, add a pledge row the way Raiser's Edge
records one (a commitment row separate from the payment against it), and read
it back. The reader drops commitment rows so pledged dollars are not counted
twice.

In [ ]:
tmp = Path(tempfile.mkdtemp())
re_export = gifts.rename(columns={
    "donor_id": "Constituent ID", "gift_date": "Gift Date", "gift_amount": "Gift Amount",
})[["Constituent ID", "Gift Date", "Gift Amount"]].assign(**{"Gift Type": "Cash"})
pledge = pd.DataFrame({
    "Constituent ID": [95515], "Gift Date": ["1995-01-01"],
    "Gift Amount": [5000.0], "Gift Type": ["Pledge"],
})
pd.concat([re_export, pledge]).to_csv(tmp / "re_gifts.csv", index=False)

raw = read_raisers_edge_gifts(tmp / "re_gifts.csv")
re_features = raisers_edge_gifts_to_features(raw, reference_date="1997-06-01")
naive = raisers_edge_gifts_to_features(raw, reference_date="1997-06-01", exclude_gift_types=None)
print(f"Donor 95515 total, pledge filtered: ${re_features.loc['95515', 'total_gift_amount']:,.0f}")
print(f"Donor 95515 total, naive sum:       ${naive.loc['95515', 'total_gift_amount']:,.0f}")
re_features[["total_gift_amount", "gift_count", "largest_gift_amount", "recency_days", "years_active"]].head()

## 4. Features as of the decision date

The mailing being predicted went out in **June 1997**. A handful of gifts in
the log are dated after that, late responses to earlier mailings. Anything
dated after the decision could not have been known when the list was pulled,
so `RFMTransformer(as_of=...)` drops it before rolling up recency, frequency
and monetary value.

In [ ]:
AS_OF = "1997-06-01"
late = gifts[gifts["gift_date"] >= AS_OF]
print(f"Gifts dated on or after {AS_OF}: {len(late)} from {late['donor_id'].nunique()} donors")

rfm = (RFMTransformer(as_of=AS_OF, include_tenure=True)
       .fit_transform(gifts[["donor_id", "gift_date", "gift_amount"]])
       .set_index("donor_id"))
rfm.head()

## 5. What leakage does to a backtest

This is the failure the library exists to prevent. Reshape the history into a
donor-period panel (one row per donor per mailing), label each row "did this
donor lapse at the next mailing", and fit `LapsePredictor` two ways:

- **as of** each period: totals built only from mailings up to that period
- **whole history**: the same totals built over the entire file, including
  mailings after the period being predicted

Both are scored with walk-forward `FiscalYearGroupedSplitter`, which never
trains on the future. A random 20,000-donor sample keeps the run to about a
minute; the full result on all 95,412 donors is in the
[real-data replication](https://philanthropy-project.github.io/PhilanthroPy/explanation/real_data_replication/).

In [ ]:
sample = donors.sample(20_000, random_state=RANDOM_STATE)
ramnt = sample[[f"RAMNT_{i}" for i in range(24, 2, -1)]].fillna(0.0).to_numpy()  # oldest first
gave = ramnt > 0
n_periods = ramnt.shape[1]

rows_as_of, rows_whole = [], []
cum_total, cum_n = np.zeros(len(sample)), np.zeros(len(sample))
for p in range(n_periods):
    cum_total, cum_n = cum_total + ramnt[:, p], cum_n + gave[:, p]
    next_gave = gave[:, p + 1] if p < n_periods - 1 else sample["TARGET_B"].to_numpy() == 1
    common = dict(period=p, recent=ramnt[:, p], lapsed=(~next_gave).astype(int))
    rows_as_of.append(pd.DataFrame(dict(total=cum_total, n=cum_n, **common)))
    rows_whole.append(pd.DataFrame(dict(total=ramnt.sum(1), n=gave.sum(1), **common)))
as_of_panel = pd.concat(rows_as_of, ignore_index=True)
whole_panel = pd.concat(rows_whole, ignore_index=True)


def backtest(panel, cv):
    panel = panel[panel["period"] < panel["period"].max()]  # hold out the final period
    X, y = panel[["total", "n", "recent"]].to_numpy(), panel["lapsed"].to_numpy()
    model = LapsePredictor(n_estimators=50, max_depth=10, random_state=RANDOM_STATE)
    groups = panel["period"].to_numpy() if isinstance(cv, FiscalYearGroupedSplitter) else None
    return cross_val_score(model, X, y, cv=cv, groups=groups, scoring="roc_auc").mean()


walk = FiscalYearGroupedSplitter(n_splits=3, drop_repeat_donors=False)
honest = backtest(as_of_panel, walk)
leaky = backtest(whole_panel, walk)
random_cv = backtest(as_of_panel, StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE))
print(f"Walk-forward, as-of features:        ROC-AUC {honest:.3f}")
print(f"Walk-forward, whole-history features: ROC-AUC {leaky:.3f}   (inflated by {leaky - honest:+.3f})")
print(f"Random K-fold, as-of features:       ROC-AUC {random_cv:.3f}   (split choice moves it {random_cv - honest:+.3f})")

The whole-history model looks better in the backtest because its totals
include gifts from after the period being predicted: it has seen part of the
answer. The real test is the final period, which neither model trained on.
Fit each on every earlier period and score that one. Both backtests
overpromise here, because the final period is the separate June 1997 mailing
rather than one more period of the same history; what matters is that the
whole-history backtest overpromises by more, and that extra is the part that
came from seeing the answer. A backtest is only useful if it tells you what
the model will do next, and the leaky one tells you the least.

In [ ]:
def held_out(panel):
    last = panel["period"].max()
    train, latest = panel[panel["period"] < last], panel[panel["period"] == last]
    model = LapsePredictor(n_estimators=50, max_depth=10, random_state=RANDOM_STATE)
    model.fit(train[["total", "n", "recent"]].to_numpy(), train["lapsed"].to_numpy())
    score = model.predict_lapse_score(latest[["total", "n", "recent"]].to_numpy())
    return roc_auc_score(latest["lapsed"], score), score


honest_future, lapse_score = held_out(as_of_panel)
leaky_future, _ = held_out(whole_panel)
print(f"As-of:         backtest promised {honest:.3f}, final period delivered {honest_future:.3f}   "
      f"(overpromised by {honest - honest_future:.3f})")
print(f"Whole-history: backtest promised {leaky:.3f}, final period delivered {leaky_future:.3f}   "
      f"(overpromised by {leaky - leaky_future:.3f})")
pd.Series(lapse_score, name="lapse score (0-100)").describe().round(1)

## 6. Who will respond to the next mailing?

One row per donor: demographics, the wealth screen (`WEALTH1`, `WEALTH2`,
`INCOME`, each missing for a quarter to half of donors), the file's own giving
summary, and the as-of RFM features from section 4. `WealthScreeningImputer`
learns its fill values from the training rows only, and `MajorGiftClassifier`
handles the remaining gaps natively.

This target is a single point in time (every donor is scored on the same
mailing), so a stratified random split is the right split here, unlike the
panel above.

In [ ]:
base_cols = ["AGE", "INCOME", "WEALTH1", "WEALTH2", "NUMCHLD", "RAMNTALL", "NGIFTALL",
             "LASTGIFT", "AVGGIFT", "MAXRAMNT", "MINRAMNT", "TIMELAG"]
X = (donors.set_index("CONTROLN")[base_cols]
     .assign(HOMEOWNER=lambda d: (donors.set_index("CONTROLN")["HOMEOWNR"] == "H").astype(int))
     .join(rfm.add_prefix("rfm_")))
y_resp = donors.set_index("CONTROLN").loc[X.index, "TARGET_B"]
y_amount = donors.set_index("CONTROLN").loc[X.index, "TARGET_D"]
gender = donors.set_index("CONTROLN").loc[X.index, "GENDER"]

X_train, X_test, yb_train, yb_test, yd_train, yd_test, g_train, g_test = train_test_split(
    X, y_resp, y_amount, gender, test_size=0.3, stratify=y_resp, random_state=RANDOM_STATE
)

response_model = make_pipeline(
    WealthScreeningImputer(wealth_cols=["WEALTH1", "WEALTH2", "INCOME"]),
    MajorGiftClassifier(random_state=RANDOM_STATE),
)
response_model.fit(X_train, yb_train)
p_respond = response_model.predict_proba(X_test)[:, 1]
affinity = response_model[-1].predict_affinity_score(response_model[:-1].transform(X_test))
print(f"Test ROC-AUC: {roc_auc_score(yb_test, p_respond):.3f}")
plot_affinity_distribution(affinity, labels=yb_test.to_numpy());

**Why does a donor score high?** Permutation importance: shuffle one column at
a time on held-out donors and measure how much the score degrades.

In [ ]:
imp_rows = X_test.sample(8_000, random_state=RANDOM_STATE).index
importance = donor_feature_importance(
    response_model, X_test.loc[imp_rows], yb_test.loc[imp_rows],
    feature_names=list(X.columns), n_repeats=3, random_state=RANDOM_STATE, scoring="roc_auc",
)
importance.head(10)

## 7. How much will they give, with an honest range?

Among donors who responded, predict gift size with `AskAmountRecommender`,
then wrap it in `GiftIntervalCalibrator`, which calibrates a 90% interval on
responders the model never saw. `interval_report` checks it on the test set.

In [ ]:
responders = yd_train > 0
X_resp, y_resp_amt = X_train[responders], yd_train[responders]
X_fit, X_cal, y_fit, y_cal = train_test_split(X_resp, y_resp_amt, test_size=0.3, random_state=RANDOM_STATE)

amount_model = AskAmountRecommender(random_state=RANDOM_STATE).fit(X_fit, y_fit)
interval_model = GiftIntervalCalibrator(amount_model, alpha=0.10, score="log").fit(X_cal, y_cal)

test_resp = yd_test > 0
interval = interval_model.predict_gift_interval(X_test[test_resp])
print(f"Certified coverage: {interval.attained_level[0]:.1%} (asked for {interval.requested_level:.0%})")
interval_report(yd_test[test_resp], interval.lower, interval.upper, alpha=0.10)

In [ ]:
ladder = amount_model.ask_ladder(X_test[test_resp].head(5), multipliers=(1.0, 1.5, 2.5))
pd.DataFrame(ladder, columns=["ask", "target", "stretch"],
             index=X_test[test_resp].head(5).index).round(2).assign(
    actual_gift=yd_test[test_resp].head(5))

## 8. Who should we mail? Decide in dollars

Each piece cost **$0.68** to mail. Mail a donor when the expected gift,
P(respond) × predicted amount, beats that cost, and compare against mailing
everyone on the held-out 30%.

In [ ]:
COST = 0.68
expected_gift = p_respond * amount_model.predict(X_test)
mail = expected_gift > COST


def campaign(mask):
    raised, cost = yd_test[mask].sum(), COST * mask.sum()
    return raised, cost


for name, mask in [("Mail everyone", np.ones(len(X_test), bool)), ("Model-targeted", mail)]:
    raised, cost = campaign(mask)
    print(f"{name:15s} pieces {mask.sum():6,d}   raised ${raised:9,.0f}   cost ${cost:8,.0f}   "
          f"net ${raised - cost:8,.0f}   ROI {fundraising_roi(total_raised=raised, total_fundraising_expense=cost):5.2f}   "
          f"cost per $ {cost_per_dollar_raised(total_fundraising_expense=cost, total_raised=raised):.2f}")

## 9. Does the list treat groups evenly?

Selection rate by recorded gender, and the four-fifths-rule ratio (below 0.8
is the usual flag). Gender is not a model input here, but proxies can carry it.

In [ ]:
mf = g_test.isin(["M", "F"]).to_numpy()
print(selection_rate_by_group(mail[mf].astype(int), g_test[mf]))
print(f"Disparate impact ratio: {disparate_impact_ratio(mail[mf].astype(int), g_test[mf]):.3f}")

## 10. Save the model for next year

`save_model` stores the fitted pipeline with its feature list, target and
library versions; `load_model` warns if you reload it under different versions.

In [ ]:
bundle_path = save_model(response_model, tmp / "response_model.joblib",
                         features=list(X.columns), target="TARGET_B")
bundle = load_model(bundle_path)
reloaded = bundle["model"]
assert np.allclose(reloaded.predict_proba(X_test)[:, 1], p_respond)
sorted(bundle)

## Where to go next

- [Avoiding temporal data leakage](https://philanthropy-project.github.io/PhilanthroPy/tutorials/avoiding_temporal_data_leakage/): the as-of idea step by step
- [Real-data replication](https://philanthropy-project.github.io/PhilanthroPy/explanation/real_data_replication/): the full leakage experiment on all 95,412 donors
- [Which estimator do I need?](https://philanthropy-project.github.io/PhilanthroPy/tutorials/): a decision table from question to class